<a href="https://colab.research.google.com/github/isoliveira20/POS-IA/blob/main/tech_challenge_02.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd
import numpy as np
import random
import matplotlib.pyplot as plt
from collections import defaultdict, Counter

teams_df = pd.read_csv('https://raw.githubusercontent.com/isoliveira20/POS-IA/refs/heads/main/tech_challenge_02/teams_dataset.csv')
tasks_df = pd.read_csv('https://raw.githubusercontent.com/isoliveira20/POS-IA/refs/heads/main/tech_challenge_02/tasks_dataset.csv')

team_info = {
    row['name']: {
        'level': row['level'],
        'stack': row['stack']
    }
    for _, row in teams_df.iterrows()
}

num_generations = 300
exploitation_prob = 0.5
population_size = len(tasks_df) * 2
sprint_days = 10
max_no_improve = 20
no_improve = 0
max_attempts_tournament = 10

base_mutation_rate = 0.05         # taxa inicial
max_mutation_rate = 0.5           # limite superior
mutation_rate = base_mutation_rate

no_improvement_threshold = 5     # quantas gerações sem melhorar antes de aumentar
best_fitness_overall = float('inf')
best_solution_overall = None

random.seed(42)
np.random.seed(42)

best_fitness_history = []
avg_fitness_history = []

level_rules = {
    'Intern': {
        'cost_multiplier': 0.5,
        'max_daily_hours': 5,
        'overload_tolerance': 0,
        'overload_penalty_multiplier': 1000,
    },
    'Junior': {
        'cost_multiplier': 0.75,
        'max_daily_hours': 7,
        'overload_tolerance': 2,
        'overload_penalty_multiplier': 500,
    },
    'Mid-level': {
        'cost_multiplier': 1.0,
        'max_daily_hours': 7,
        'overload_tolerance': 5,
        'overload_penalty_multiplier': 200,
    },
    'Senior': {
        'cost_multiplier': 1.25,
        'max_daily_hours': 7,
        'overload_tolerance': 10,
        'overload_penalty_multiplier': 100,
    },
    'Specialist': {
        'cost_multiplier': 1.5,
        'max_daily_hours': 7,
        'overload_tolerance': 10,
        'overload_penalty_multiplier': 100,
    },
}



priority_weight = {'Disaster': 5, 'Critical': 4, 'High':3, 'Medium':2, 'Low':1}

developers = teams_df['name'].tolist()

# dev é compatível
def is_compatible(dev_name, task_stack, teams_df):
    dev_stack = teams_df.loc[teams_df['name'] == dev_name, 'stack'].values[0]
    return dev_stack == task_stack

# geração para criar solução válida com priorização de tasks
def generate_random_chromosome_prioritized(tasks_df, developers, teams_df):
    chromosome = []
    # Ordena tasks da maior para menor prioridade
    tasks_sorted = tasks_df.copy()
    tasks_sorted['priority_weight'] = tasks_sorted['priority'].map(priority_weight)
    tasks_sorted = tasks_sorted.sort_values(by='priority_weight', ascending=False)

    for _, task_row in tasks_sorted.iterrows():
        task_stack = task_row['stack']
        # Devs compatíveis
        compatible_devs = [dev for dev in developers if teams_df.loc[teams_df['name'] == dev, 'stack'].values[0] == task_stack]

        if compatible_devs:
            assigned_dev = random.choice(compatible_devs)
        else: # fallback para qualquer dev
            assigned_dev = random.choice(developers)

        chromosome.append(assigned_dev)

    return chromosome

# gerar população inicial
def generate_random_chromosome_balanced(tasks_df, developers, teams_df):
    dev_hours = {dev: 0 for dev in developers}
    chromosome = []

    for _, task_row in tasks_df.iterrows():
        task_stack = task_row['stack']
        hours = task_row['estimated_hours']
        compatible_devs = [dev for dev in developers if teams_df.loc[teams_df['name']==dev, 'stack'].values[0]==task_stack]
        if compatible_devs:
            # escolhe dev compatível com menor carga atual
            assigned_dev = min(compatible_devs, key=lambda d: dev_hours[d])
        else:
            assigned_dev = min(developers, key=lambda d: dev_hours[d])

        chromosome.append(assigned_dev)
        dev_hours[assigned_dev] += hours

    return chromosome


def generate_random_chromosome_experienced(tasks_df, developers, teams_df):
    chromosome = []
    for _, task in tasks_df.iterrows():
        stack = task['stack']
        priority = task['priority']
        # Para tarefas críticas, priorize devs Senior/Specialist
        if priority in ['Disaster', 'Critical']:
            compatible_devs = [
                dev for dev in developers
                if teams_df.loc[teams_df['name'] == dev, 'stack'].values[0] == stack
                and teams_df.loc[teams_df['name'] == dev, 'level'].values[0] in ['Senior', 'Specialist']
            ]
        else:
            compatible_devs = [dev for dev in developers if teams_df.loc[teams_df['name'] == dev, 'stack'].values[0] == stack]
        if compatible_devs:
            assigned = random.choice(compatible_devs)
        else:
            assigned = random.choice(developers)
        chromosome.append(assigned)
    return chromosome


def generate_population(population_size, tasks_df, developers, teams_df):
    population_new = []
    for _ in range(population_size // 3):
        population_new.append(generate_random_chromosome_prioritized(tasks_df, developers, teams_df))
    for _ in range(population_size // 3):
        population_new.append(generate_random_chromosome_balanced(tasks_df, developers, teams_df))
    for _ in range(population_size - 2*(population_size // 3)):
        population_new.append(generate_random_chromosome_experienced(tasks_df, developers, teams_df))

    return population_new


def fitness(chromosome, tasks_df, teams_df):
    dev_hours = {dev: 0 for dev in teams_df['name']}
    dev_cost = {dev: 0 for dev in teams_df['name']}
    penalty = 0.0

    tasks_by_level = defaultdict(int)

    for task_idx, dev in enumerate(chromosome):
        task = tasks_df.iloc[task_idx]
        hours = task['estimated_hours']
        dev_info = team_info[dev]
        level = dev_info['level']
        stack = dev_info['stack']
        priority = task['priority']

        dev_hours[dev] += hours
        cost = hours * level_rules[level]['cost_multiplier']
        dev_cost[dev] += cost
        tasks_by_level[level] += 1

        priority_w = priority_weight.get(priority, 1)

        # Penalidades reescaladas
        if stack != task['stack']:
            penalty += 2 * priority_w    # antes 20 * priority_w

        if level in ['Intern', 'Junior'] and priority in ['Critical', 'Disaster']:
            penalty += 20   # antes 200

    total_cost = sum(dev_cost.values())

    overload_score = 0.0
    for dev, hours in dev_hours.items():
        dev_info = teams_df[teams_df['name'] == dev].iloc[0]
        level = dev_info['level']
        max_hours = level_rules[level]['max_daily_hours'] * sprint_days

        if hours > max_hours:
            overload = hours - level_rules[level]['max_daily_hours']
            tolerance = level_rules[level]['overload_tolerance']
            critical_overload = max(0, overload - tolerance)
            penalty += (level_rules[level]['overload_penalty_multiplier'] / 10) * critical_overload  # ex: 1000→100

            overload_score += overload / max_hours

            if level == 'Intern':
                penalty += 10 * overload    # antes 100

    hours_list = np.array(list(dev_hours.values()))
    avg_hours = np.mean(hours_list) if len(hours_list) > 0 else 0.0
    std_hours = np.std(hours_list)

    MIN_AVG_HOURS = 1e-1
    safe_avg_hours = max(avg_hours, MIN_AVG_HOURS)
    imbalance_score = std_hours / safe_avg_hours

    level_task_counts = [tasks_by_level[lvl] for lvl in ['Intern', 'Junior', 'Mid-level', 'Senior', 'Specialist']]
    distribution_penalty = np.std(level_task_counts)

    w_overload = 0.6
    w_cost = 0.08
    w_imbalance = 0.4
    w_distribution = 0.15
    w_base_penalties = 0.1

    fitness_value = (
        w_overload * overload_score +
        w_cost * total_cost +
        w_imbalance * imbalance_score +
        w_distribution * distribution_penalty +
        w_base_penalties * penalty
    )

    return fitness_value


# escolhendo os pais por torneio
def tournament_selection(population, fitnesses, k=7):
    selected = random.sample(list(zip(population, fitnesses)), k)
    selected.sort(key=lambda x: x[1])  # Menor fitness é melhor

    return selected[0][0]


# corrigir incompatibilidades
def correct_incompatibilities(chromosome, tasks_df, developers, teams_df):
    corrected_chromosome = chromosome.copy()
    task_counts = Counter(chromosome)

    for i, dev in enumerate(corrected_chromosome):
        task_stack = tasks_df.iloc[i]['stack']
        dev_stack = team_info[dev]['stack']

        if dev_stack != task_stack:
            compatible_devs = [
                d for d in developers if team_info[d]['stack'] == task_stack
            ]

            if compatible_devs:
                # Preferir quem tem menos tarefas no cromossomo original
                chosen_dev = min(compatible_devs, key=lambda d: task_counts[d])

                # Atualiza o cromossomo e os contadores
                old_dev = corrected_chromosome[i]
                corrected_chromosome[i] = chosen_dev

                task_counts[old_dev] -= 1
                task_counts[chosen_dev] += 1
            else:
                # Sem dev compatível — mantém o original
                pass

    return corrected_chromosome

def uniform_crossover(parent1, parent2):
    child = []

    for a, b in zip(parent1, parent2):
      child.append(random.choice([a, b]))

    return child

def two_point_crossover(parent1, parent2):
    size = len(parent1)

    pt1, pt2 = sorted(random.sample(range(size), 2))

    child = parent1[:pt1] + parent2[pt1:pt2] + parent1[pt2:]

    return child

# mutação
def mutate_balanced(chromosome, developers, teams_df, tasks_df, mutation_rate):
    dev_hours = {dev: 0 for dev in developers}

    if mutation_rate > base_mutation_rate:
        exploitation_prob_adap = max(0.2, exploitation_prob - 0.05)
    else:
        exploitation_prob_adap = min(0.9, exploitation_prob + 0.05)

    # Calcula carga atual
    for i, dev in enumerate(chromosome):
        dev_hours[dev] += tasks_df.iloc[i]['estimated_hours']

    for i in range(len(chromosome)):
        if random.random() < mutation_rate:
            task_stack = tasks_df.iloc[i]['stack']
            task_hours = tasks_df.iloc[i]['estimated_hours']

            compatible_devs = [
                dev for dev in developers
                if teams_df.loc[teams_df['name'] == dev, 'stack'].values[0] == task_stack
            ]

            if compatible_devs:
                if random.random() < exploitation_prob_adap:
                    assigned_dev = min(compatible_devs, key=lambda d: dev_hours[d])
                else:
                    other_devs = [d for d in compatible_devs if d != chromosome[i]]
                    if not other_devs:
                        continue
                    assigned_dev = random.choice(other_devs)

                new_hours = dev_hours[assigned_dev] + task_hours
                level = teams_df.loc[teams_df['name'] == assigned_dev, 'level'].values[0]
                max_allowed = level_rules[level]['max_daily_hours'] * sprint_days

                if new_hours <= max_allowed:
                    dev_hours[assigned_dev] += task_hours
                    dev_hours[chromosome[i]] -= task_hours
                    chromosome[i] = assigned_dev

    return chromosome


def validate_chromosome(chromosome, tasks_df, teams_df):
    for i, dev in enumerate(chromosome):
        task_stack = tasks_df.iloc[i]['stack']
        dev_stack = team_info[dev]['stack']
        if dev_stack != task_stack:
            return False

    return True

def print_dev_load(chromosome, tasks_df):
    dev_hours = defaultdict(float)

    for task_idx, dev in enumerate(chromosome):
        dev_hours[dev] += tasks_df.iloc[task_idx]['estimated_hours']

    dev_hours = sorted(dev_hours.items(), key=lambda x: x[0], reverse=False)
    print("Carga horária por desenvolvedor:")
    for dev, hours in dev_hours:
        dev_info = teams_df[teams_df['name'] == dev].iloc[0]
        tasks = filter_tasks_per_dev(chromosome, dev)
        print(f"{dev}: {hours:.2f}h ({tasks} tasks) - {dev_info['level']} ({dev_info['stack']})")

def check_compatibility(chromosome, tasks_df, teams_df):
    incompatibilities = []
    for i, dev in enumerate(chromosome):
        task_stack = tasks_df.iloc[i]['stack']
        dev_stack = teams_df.loc[teams_df['name'] == dev, 'stack'].values[0]
        if dev_stack != task_stack:
            incompatibilities.append((i, dev, task_stack, dev_stack))
    if incompatibilities:
        print("Incompatibilidades encontradas:")
        for task_idx, dev, task_stack, dev_stack in incompatibilities:
            print(f"Task {task_idx} (stack {task_stack}) -> Dev {dev} (stack {dev_stack})")
    else:
        print("Todas as tasks estão alocadas a devs compatíveis!")


def tasks_per_dev(chromosome):
    count = Counter(chromosome)
    print("Número de tasks por dev:")
    for dev, num in count.items():
        print(f"{dev}: {num}")


def filter_tasks_per_dev(chromosome, dev):
    count = Counter(chromosome)
    return count.get(dev, 0)

def local_search_bidirectional_swaps(chromosome, tasks_df, teams_df, developers, max_iterations=5, max_swaps_per_iter=100):
    best_chr = chromosome
    best_fit = fitness(chromosome, tasks_df, teams_df)
    iteration = 0
    improved = True
    n_tasks = len(chromosome)

    while improved and iteration < max_iterations:
        improved = False
        swaps_tested = 0
        pairs = [(i, j) for i in range(n_tasks) for j in range(i+1, n_tasks)]
        random.shuffle(pairs)

        for i, j in pairs:
            if swaps_tested >= max_swaps_per_iter:
                break
            swaps_tested += 1

            dev_i = best_chr[i]
            dev_j = best_chr[j]
            task_i = tasks_df.iloc[i]
            task_j = tasks_df.iloc[j]

            if (team_info[dev_j]['stack'] == task_i['stack'] and
                team_info[dev_i]['stack'] == task_j['stack']):

                new_chr = best_chr.copy()
                new_chr[i] = dev_j
                new_chr[j] = dev_i

                dev_hours = {dev: 0 for dev in developers}
                for idx, dev in enumerate(new_chr):
                    dev_hours[dev] += tasks_df.iloc[idx]['estimated_hours']

                max_hours_i = level_rules[team_info[dev_i]['level']]['max_daily_hours'] * sprint_days
                max_hours_j = level_rules[team_info[dev_j]['level']]['max_daily_hours'] * sprint_days

                if dev_hours[dev_i] <= max_hours_i and dev_hours[dev_j] <= max_hours_j:
                    new_fit = fitness(new_chr, tasks_df, teams_df)
                    if new_fit < best_fit:
                        best_fit = new_fit
                        best_chr = new_chr
                        improved = True
                        break
        iteration += 1
    return best_chr


def local_search_incremental(chromosome, tasks_df, teams_df, developers, max_iterations=3, max_swaps_per_iter=50):
    best_chr = chromosome
    best_fit = fitness(chromosome, tasks_df, teams_df)
    iteration = 0
    improved = True
    n_tasks = len(chromosome)

    # Calcula cargas iniciais uma vez só
    dev_hours = {dev:0 for dev in developers}
    for idx, dev in enumerate(chromosome):
        dev_hours[dev] += tasks_df.iloc[idx]['estimated_hours']

    while improved and iteration < max_iterations:
        improved = False
        swaps_tested = 0

        # Amostra aleatória de pares (rapidamente explora espaço sem rodar todos)
        all_pairs = [(i, j) for i in range(n_tasks) for j in range(i+1, n_tasks)]
        random.shuffle(all_pairs)

        for i, j in all_pairs:
            if swaps_tested >= max_swaps_per_iter:
                break
            swaps_tested += 1

            dev_i = best_chr[i]
            dev_j = best_chr[j]
            task_i = tasks_df.iloc[i]['estimated_hours']
            task_j = tasks_df.iloc[j]['estimated_hours']

            # Compatibilidade para swap
            if (team_info[dev_j]['stack'] == tasks_df.iloc[i]['stack'] and
                team_info[dev_i]['stack'] == tasks_df.iloc[j]['stack']):

                # Atualiza as cargas apenas nos devs do swap
                new_dev_hours = dev_hours.copy()
                new_dev_hours[dev_i] += task_j - task_i
                new_dev_hours[dev_j] += task_i - task_j

                max_hours_i = level_rules[team_info[dev_i]['level']]['max_daily_hours'] * sprint_days
                max_hours_j = level_rules[team_info[dev_j]['level']]['max_daily_hours'] * sprint_days

                if new_dev_hours[dev_i] <= max_hours_i and new_dev_hours[dev_j] <= max_hours_j:
                    new_chr = best_chr.copy()
                    new_chr[i] = dev_j
                    new_chr[j] = dev_i

                    new_fit = fitness(new_chr, tasks_df, teams_df)
                    if new_fit < best_fit:
                        best_chr = new_chr
                        best_fit = new_fit
                        dev_hours = new_dev_hours  # só atualiza se swap foi bom
                        improved = True
                        break
        iteration += 1
    return best_chr


population = generate_population(population_size, tasks_df, developers, teams_df)

print(f"População inicial gerada com {population_size} soluções.")
print("Exemplo de cromossomo da população:")
print(population[0])

for generation in range(num_generations):
    # Avaliar fitness de cada indivíduo
    fitness_values = [fitness(individual, tasks_df, teams_df) for individual in population]

    best_fitness = min(fitness_values)
    avg_fitness = sum(fitness_values) / len(fitness_values)
    best_fitness_history.append(best_fitness)
    avg_fitness_history.append(avg_fitness)

    print(f"Geração {generation + 1} - Melhor fitness: {best_fitness:.2f} - Fitness médio: {avg_fitness:.2f}")

    if best_fitness < best_fitness_overall:
        if mutation_rate != base_mutation_rate:
            print(f"Resetando mutação")
            mutation_rate = base_mutation_rate

        no_improve = 0

        best_fitness_overall = best_fitness
        best_solution_overall = population[np.argmin(fitness_values)]
    else:
        no_improve += 1

        if no_improve >= no_improvement_threshold:
            inc_mutation_rate = min(mutation_rate + 0.05, max_mutation_rate)

            if inc_mutation_rate != mutation_rate:
                mutation_rate = inc_mutation_rate
                print(f"Aumentando mutação para: {mutation_rate}")

    if no_improve >= max_no_improve:
        print("Parando por estagnação!")
        break

    new_population = []

    # Elite
    elite_size = 3  # Escolhe só o top 3
    sorted_pop = [x for _, x in sorted(zip(fitness_values, population), key=lambda pair: pair[0])]
    elite = sorted_pop[:elite_size]

    # Aplicação de busca local só nos top 3 elite
    new_population.extend([local_search_incremental(ind, tasks_df, teams_df, developers) for ind in elite])

    # validar alterar p 30 a 70 gerações
    if generation % 30 == 0:
        for _ in range(population_size // 10):
            new_population.append(generate_random_chromosome_balanced(tasks_df, developers, teams_df))

    # Criação de nova população até completar o tamanho
    while len(new_population) < population_size:
        # Seleção dos pais via torneio
        parent1 = tournament_selection(population, fitness_values)
        parent2 = tournament_selection(population, fitness_values)

        attempts_tournament = 0
        while parent1 == parent2 and attempts_tournament < max_attempts_tournament:
            parent2 = tournament_selection(population, fitness_values)
            attempts_tournament += 1

        if parent1 == parent2:
            # fallback simples: escolhe qualquer um diferente do parent1
            candidates = [ind for ind in population if ind != parent1]
            parent2 = random.choice(candidates)

        # Gerar filho pelo crossover e corrigir incompatibilidades
        if random.random() < 0.5:
            child = two_point_crossover(parent1, parent2)
        else:
            child = uniform_crossover(parent1, parent2)

        child = correct_incompatibilities(child, tasks_df, developers, teams_df)

        # Mutação
        child = mutate_balanced(child, developers, teams_df, tasks_df, mutation_rate)
        new_population.append(child)

    population = new_population


print("Melhor solução encontrada:")
print(best_solution_overall)

# Análises complementares
print_dev_load(best_solution_overall, tasks_df)
check_compatibility(best_solution_overall, tasks_df, teams_df)

plt.plot(best_fitness_history, label='Melhor fitness')
plt.plot(avg_fitness_history, label='Fitness médio')
plt.xlabel('Geração')
plt.ylabel('Fitness')
plt.title('Evolução do Fitness')
plt.legend()
plt.grid(True)
plt.show()

def get_dev_tasks(chromosome, tasks_df):
    dev_task_map = defaultdict(list)
    for task_idx, dev in enumerate(chromosome):
        task_info = tasks_df.iloc[task_idx]
        dev_task_map[dev].append({
            "task_idx": task_idx,
            "stack": task_info['stack'],
            "priority": task_info['priority'],
            "estimated_hours": task_info['estimated_hours'],
            "task_name": task_info.get('name', f"Task_{task_idx+1}")
        })
    return dev_task_map

def print_dev_task_details(chromosome, tasks_df):
    dev_tasks = get_dev_tasks(chromosome, tasks_df)
    for dev, tasks in dev_tasks.items():
        total_hours = sum(t['estimated_hours'] for t in tasks)
        print(f"\n{dev}: {total_hours:.2f}h, {len(tasks)} tasks")
        for t in tasks:
            print(f"  - {t['task_name']}: {t['estimated_hours']}h [{t['stack']}/{t['priority']}]")


print_dev_task_details(best_solution_overall, tasks_df)

import matplotlib.pyplot as plt
dev_hours = defaultdict(float)
for task_idx, dev in enumerate(best_solution_overall):
    dev_hours[dev] += tasks_df.iloc[task_idx]['estimated_hours']
plt.bar(dev_hours.keys(), dev_hours.values())
plt.xticks(rotation=90)
plt.ylabel('Horas Totais')
plt.title('Horas Alocadas por Dev')
plt.show()

População inicial gerada com 198 soluções.
Exemplo de cromossomo da população:
['F_Dev1', 'A_Dev8', 'A_Dev1', 'C_Dev1', 'B_Dev1', 'B_Dev1', 'B_Dev2', 'F_Dev1', 'A_Dev8', 'F_Dev3', 'A_Dev6', 'A_Dev3', 'B_Dev4', 'B_Dev4', 'A_Dev5', 'B_Dev4', 'B_Dev7', 'E_Dev3', 'D_Dev1', 'B_Dev4', 'E_Dev3', 'B_Dev6', 'F_Dev1', 'F_Dev1', 'E_Dev2', 'E_Dev3', 'D_Dev4', 'B_Dev7', 'E_Dev1', 'D_Dev1', 'C_Dev1', 'A_Dev1', 'B_Dev1', 'D_Dev4', 'C_Dev3', 'C_Dev1', 'B_Dev2', 'B_Dev6', 'C_Dev3', 'A_Dev8', 'B_Dev4', 'D_Dev3', 'B_Dev4', 'C_Dev6', 'C_Dev5', 'C_Dev1', 'A_Dev2', 'E_Dev1', 'F_Dev3', 'A_Dev8', 'D_Dev5', 'A_Dev6', 'D_Dev1', 'C_Dev2', 'B_Dev5', 'E_Dev3', 'B_Dev6', 'F_Dev1', 'A_Dev3', 'A_Dev3', 'B_Dev7', 'C_Dev2', 'A_Dev6', 'B_Dev7', 'B_Dev4', 'D_Dev5', 'C_Dev1', 'D_Dev5', 'C_Dev5', 'B_Dev3', 'C_Dev5', 'D_Dev3', 'A_Dev5', 'F_Dev1', 'D_Dev3', 'E_Dev2', 'A_Dev6', 'B_Dev3', 'A_Dev5', 'B_Dev1', 'E_Dev1', 'D_Dev3', 'A_Dev3', 'D_Dev1', 'E_Dev2', 'A_Dev5', 'F_Dev1', 'C_Dev3', 'A_Dev4', 'B_Dev1', 'A_Dev2', 'C_Dev4', 